**Clock v2 note.** The code this notebook runs now lives in `src/` only. Before importing anything from `src`, point it at Drive with

```
%env CDT_DATA_DIR=/content/drive/MyDrive/cdt-data
%env CDT_MODELS_DIR=/content/drive/MyDrive/cdt-data/models
```

Setting `cfg.LABELS_CSV = ...` after the import does not reach `train.py`, which is why the old Drive path was hardcoded there. The outputs saved below are from the v1 run and need to be re-run.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content
import os
if not os.path.exists('/content/cdt-alzheimer-screening'):
    !git clone https://github.com/wiambenadder/cdt-alzheimer-screening.git
%cd cdt-alzheimer-screening
!git pull 2>/dev/null
!pip install -q -r requirements.txt
import sys
sys.path.insert(0, '/content/cdt-alzheimer-screening')
print("setup ok")

Mounted at /content/drive
/content
Cloning into 'cdt-alzheimer-screening'...
remote: Enumerating objects: 142, done.
remote: Counting objects: 100% (142/142), done.
remote: Compressing objects: 100% (129/129), done.
remote: Total 142 (delta 72), reused 27 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (142/142), 672.08 KiB | 3.29 MiB/s, done.
Resolving deltas: 100% (72/72), done.
/content/cdt-alzheimer-screening
Already up to date.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 105.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 152.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 112.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 94.8 MB/s eta 0:00:00
set

In [2]:
# Re-stage all rounds for error analysis on full test set
import shutil, time
from pathlib import Path

LOCAL_DATA_DIR = Path('/content/nhats_local')
LOCAL_DATA_DIR.mkdir(exist_ok=True)
DRIVE_DATA_DIR = Path('/content/drive/MyDrive/cdt-data/nhats_raw')

t0 = time.time()
for rf in sorted(DRIVE_DATA_DIR.iterdir()):
    if not rf.is_dir(): continue
    dst = LOCAL_DATA_DIR / rf.name
    if dst.exists() and len(list(dst.glob('*.tif'))) > 0:
        continue
    print(f"[copy] {rf.name}")
    shutil.copytree(rf, dst, dirs_exist_ok=True)
print(f"staged in {(time.time()-t0)/60:.1f} min")

import pandas as pd
drive_labels = pd.read_csv('/content/drive/MyDrive/cdt-data/labels.csv')
drive_labels['image_path'] = drive_labels['image_path'].str.replace(
    '/content/drive/MyDrive/cdt-data/nhats_raw/',
    '/content/nhats_local/', regex=False)
drive_labels.to_csv('/content/cdt_labels_local.csv', index=False)
print(f"Loaded {len(drive_labels):,} labels")

[copy] round_01
[copy] round_02
[copy] round_03
[copy] round_04
[copy] round_05
[copy] round_06
[copy] round_07
[copy] round_08
[copy] round_09
[copy] round_10
[copy] round_11
[copy] round_12
[copy] round_13
[copy] round_14
staged in 42.5 min
Loaded 59,417 labels


*This cell used to overwrite `src/data.py` with `%%writefile`. That code now lives in `src/data.py` directly, so the cell was removed.*

In [5]:
import importlib
import src.config, src.data, src.augmentation, src.models, src.evaluate, src.utils
for m in [src.config, src.data, src.augmentation, src.models, src.evaluate, src.utils]:
    importlib.reload(m)

import src.config as cfg
cfg.LABELS_CSV  = Path('/content/cdt_labels_local.csv')
cfg.MODELS_DIR  = Path('/content/drive/MyDrive/cdt-data/models')
cfg.RESULTS_DIR = Path('/content/drive/MyDrive/cdt-data/results')

from src.data import get_splits, CDTDataset
from src.config import DataConfig
from src.augmentation import build_eval_transform
from torch.utils.data import DataLoader

df = pd.read_csv(cfg.LABELS_CSV)
df = df.rename(columns={'cdt_score': 'label'})[['participant_id', 'label', 'image_path']]
df['participant_id'] = df['participant_id'].astype(str)
data_cfg = DataConfig(batch_size=32, num_workers=4)

# Same SEED → same splits as notebook 03
train_df, val_df, test_df = get_splits(df, data_cfg)

eval_tf = build_eval_transform(data_cfg.image_size)
test_ds = CDTDataset(test_df, transform=eval_tf)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=4,
                         pin_memory=True, persistent_workers=True, prefetch_factor=4)
print(f"\n[ok] test set: {len(test_df):,} clocks ready for analysis")

[split] participant-disjoint:
  train: 41,551 clocks from 9,549 participants
  val:   8,928 clocks from 2,046 participants
  test:  8,938 clocks from 2,047 participants

[ok] test set: 8,938 clocks ready for analysis


In [6]:
import torch
from src.models import get_model
from src.utils import load_checkpoint, get_device

device = get_device()
ckpt_path = cfg.MODELS_DIR / 'vit_b16_ft_best.pt'
print(f"Loading: {ckpt_path}")

model, _, _ = get_model('vit_b16', freeze_backbone=False, dropout=0.3)
model = load_checkpoint(model, str(ckpt_path), device=device).to(device)
model.eval()
print(f"[ok] ViT-B/16 loaded ({sum(p.numel() for p in model.parameters()):,} params)")

Loading: /content/drive/MyDrive/cdt-data/models/vit_b16_ft_best.pt
Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:01<00:00, 246MB/s]


[ok] ViT-B/16 loaded (85,803,270 params)


In [7]:
from src.evaluate import collect_predictions
import numpy as np

print("Running ViT over test set...")
y_true, y_pred, y_probs = collect_predictions(model, test_loader, device)
print(f"Predictions: {len(y_pred):,}")
print(f"Distribution of predicted classes: {np.bincount(y_pred, minlength=6).tolist()}")
print(f"Distribution of true classes:      {np.bincount(y_true, minlength=6).tolist()}")

Running ViT over test set...
Predictions: 8,938
Distribution of predicted classes: [103, 389, 1288, 1791, 2599, 2768]
Distribution of true classes:      [94, 323, 996, 1976, 3187, 2362]


In [8]:
from src.evaluate import plot_confusion_matrix, save_classification_report

# Raw counts
plot_confusion_matrix(y_true, y_pred,
                      cfg.RESULTS_DIR / 'vit_confusion_matrix.png',
                      normalize=False)
print(f"saved: vit_confusion_matrix.png")

# Normalized (per-class recall) — easier to interpret
plot_confusion_matrix(y_true, y_pred,
                      cfg.RESULTS_DIR / 'vit_confusion_matrix_normalized.png',
                      normalize=True)
print(f"saved: vit_confusion_matrix_normalized.png")

# Detailed per-class report
save_classification_report(y_true, y_pred,
                           cfg.RESULTS_DIR / 'vit_classification_report.json')

# Print human-readable version
from sklearn.metrics import classification_report
print("\n" + classification_report(y_true, y_pred,
    labels=list(range(6)),
    target_names=['0_severe', '1_significant', '2_moderate', '3_mild_mod',
                  '4_mild', '5_normal'],
    zero_division=0))

saved: vit_confusion_matrix.png
saved: vit_confusion_matrix_normalized.png

               precision    recall  f1-score   support

     0_severe       0.57      0.63      0.60        94
1_significant       0.59      0.71      0.64       323
   2_moderate       0.53      0.69      0.60       996
   3_mild_mod       0.69      0.62      0.65      1976
       4_mild       0.74      0.61      0.67      3187
     5_normal       0.68      0.79      0.73      2362

     accuracy                           0.67      8938
    macro avg       0.63      0.67      0.65      8938
 weighted avg       0.68      0.67      0.67      8938



In [9]:
from src.evaluate import analyze_errors
from pathlib import Path

errors_dir = cfg.RESULTS_DIR / 'errors'
errors_dir.mkdir(parents=True, exist_ok=True)

records = analyze_errors(
    y_true, y_pred, y_probs,
    df_test=test_df.reset_index(drop=True),
    save_dir=errors_dir,
    k=12,
)
print(f"\nSaved {len(records)} error examples to {errors_dir}/")
print("\nTop misclassifications:")
import pandas as pd
print(pd.DataFrame(records)[['true_label', 'pred_label', 'pred_confidence', 'ordinal_distance']]
      .head(12).to_string())


Saved 12 error examples to /content/drive/MyDrive/cdt-data/results/errors/

Top misclassifications:
    true_label  pred_label  pred_confidence  ordinal_distance
0            1           0         0.996119                 1
1            3           1         0.988242                 2
2            3           2         0.982709                 1
3            3           2         0.981279                 1
4            2           3         0.976093                 1
5            3           2         0.976035                 1
6            1           2         0.973787                 1
7            3           2         0.973283                 1
8            3           2         0.970919                 1
9            0           1         0.970352                 1
10           3           2         0.969810                 1
11           3           2         0.969575                 1


In [10]:
from src.evaluate import measure_inference_time

print("Measuring inference throughput...")
timing = measure_inference_time(model, test_loader, device, n_batches=20)
print(f"\nViT-B/16 inference: {timing['ms_per_image']:.1f} ms/image")
print(f"                    {timing['images_per_sec']:.0f} images/sec")
print(f"\nThis matters for clinical deployment - a clock takes <1 second to score.")

Measuring inference throughput...

ViT-B/16 inference: 2.3 ms/image
                    436 images/sec

This matters for clinical deployment - a clock takes <1 second to score.


In [ ]:
from src.gradcam import plot_attribution_grid
from src.augmentation import build_eval_transform

# the heatmap is drawn on the model's own input (crop + resize), so they line up.
# use crop_method="projection" for the v1 checkpoints, "component" for v2 ones
eval_tf = build_eval_transform(224, crop_method="projection")
plot_attribution_grid(model, test_df, y_true, y_pred, eval_tf, device,
                      save_path=cfg.RESULTS_DIR / "vit_attribution_maps.png")


In [12]:
import pandas as pd
import json

results = {}
for fn in ['baselines.json', 'finetuning_results.json', 'ablation_results.json']:
    p = cfg.RESULTS_DIR / fn
    if p.exists():
        with open(p) as f:
            results.update(json.load(f))

rows = []
for name, m in results.items():
    if isinstance(m, dict):
        rows.append({
            'run': name,
            'accuracy': m.get('accuracy'),
            'macro_f1': m.get('macro_f1'),
            'quadratic_kappa': m.get('quadratic_kappa'),
            'macro_auc': m.get('macro_auc'),
            'note': m.get('note', ''),
        })

df_all = pd.DataFrame(rows).set_index('run')
df_all = df_all.sort_values('quadratic_kappa', ascending=False, na_position='last')
df_all.to_csv(cfg.RESULTS_DIR / 'all_results.csv')
print("=" * 100)
print("ALL RESULTS — sorted by quadratic kappa")
print("=" * 100)
print(df_all.round(4).to_string())
print(f"\nSaved to {cfg.RESULTS_DIR / 'all_results.csv'}")

ALL RESULTS — sorted by quadratic kappa
                    accuracy  macro_f1  quadratic_kappa  macro_auc                                                note
run                                                                                                                   
abl_unfrozen_aug      0.6760    0.6555           0.8117     0.9257                                                    
vit_b16_ft            0.6719    0.6484           0.8085     0.9253                                                    
abl_unfrozen_noaug    0.6457    0.6227           0.7992     0.9156                                                    
vgg16_ft              0.6160    0.5842           0.7813     0.9026  trained 8 epochs, stopped manually for time budget
effb0_ft              0.5895    0.5493           0.7465     0.8932                                                    
abl_frozen_noaug      0.5015    0.4942           0.6670     0.8489                                                    
abl_froz